In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [3]:
loader = PyMuPDFLoader("수업 데이터/wt_data.pdf")
docs = loader.load()

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

split_docs = splitter.split_documents(docs)

In [5]:
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name="student_support",
    persist_directory="./student_chroma_db",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [6]:
@tool
def retrieve_context(query: str):
    """학교에 관련된 정보를 검색합니다."""
    retrieved_docs = retriever.invoke(query)

    text_data = ""
    for i, doc in enumerate(retrieved_docs):
        text_data += f"[문서 {i+1}] \n"
        text_data += f"source: {doc.metadata.get('source')} \n"
        text_data += f"page : {doc.metadata.get('page')} \n "
        text_data += f"content : {doc.page_content}"

    return text_data, retrieved_docs

In [7]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[retrieve_context],
    system_prompt=(
        "당신은 학생지원센터 안내를 담당하고 있는 에이전트입니다."
        "반드시 retieve_context 도구를 먼저 사용하여 관련 내용을 문서에서 찾은뒤 답변하세요"
        '문서에 없는 내용은 추측해서 답변하지 말고 "음...난 잘 알지못해요"라고 답하세요'
        "가능하며 답변끝에 참고한 페이지 번호등의 출처를 제시하세요"
    ),
)

In [8]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "상담 예약은 언제까지 취소할수 있나요?"}]}
)

print(result["messages"][-1].content)

상담 예약 취소 가능 시점은 아래와 같습니다.

- 일반 취소: 상담 시작 4시간 전까지 취소 가능. 이 경우 불이익이 없습니다.
- 지연 취소: 상담 시작 4시간 전 이후 ~ 시작 전까지 취소도 가능하나, 정책에 따라 월 2회까지는 경고가 면제됩니다(즉, 2회까지는 경고 없이 취소가 가능합니다).

노쇼(사전 연락 없이 미참석) 정책
- 노쇼: 사전 연락 없이 미참석한 경우 해당 학기의 온라인 예약이 2주 제한됩니다.
- 반복 노쇼: 같은 학기 내에 2회 이상 노쇼 시 상담사 사전 승인 후에만 예약 가능.

예외 조항
- 병원 진료 확인서, 가족 장례, 학교 공식 공결 등 증빙이 가능한 사유가 있으면 노쇼로 처리하지 않을 수 있습니다. 다만 증빙은 상담일로부터 3영업일 이내에 제출해야 합니다.

참고 출처
- 수업 데이터/wt_data.pdf, 페이지 3(2. 상담 예약 및 취소 규정) 및 페이지 2-4(노쇼 정책 및 예외 조항)
